In [1]:
from langchain_community.document_loaders import PyPDFLoader
import chromadb
from pathlib import Path

# 1. Setup paths and parameters
PDF_PATH = "./documents"
DB_DIR = "./chromadb"

# Initialize your Chroma vector store
chroma_client = chromadb.PersistentClient(path=DB_DIR)

collection = chroma_client.get_or_create_collection(name="religious_collection")
# Specify the directory path (use "." for current directory)
directory = Path(PDF_PATH)

# List all .pdf files in the directory
pdf_files = list(directory.glob("*.pdf"))

for afile in pdf_files:
    print(f"Processing file: {afile.name}")
    # 1. Load the PDF - PyPDFLoader automatically separates text per page
    loader = PyPDFLoader(str(afile))
    pages = loader.load_and_split()

    for apage in pages:
        page_text = str(apage.page_content)
        page_id = str(apage.metadata['source'])
        page_id = page_id + "_" + str(apage.metadata['page'])
        
        # 2. Add documents to the Chroma vector store
        collection.add(documents=[page_text], ids=[page_id])
        
    print(f"Loaded {len(pages)} pages from {afile.name}.")


C:\Users\SowmyaVenky\AppData\Local\Temp\ipykernel_10444\182995905.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Processing file: atma_bodha.pdf
Loaded 106 pages from atma_bodha.pdf.
Processing file: bgita.pdf


Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 224 0 (offset 0)


Loaded 154 pages from bgita.pdf.
Processing file: bgita_1.pdf
Loaded 951 pages from bgita_1.pdf.
Processing file: tattva_bodha.pdf
Loaded 96 pages from tattva_bodha.pdf.


In [2]:
import chromadb
chroma_client = chromadb.PersistentClient(path=DB_DIR)
collections = chroma_client.list_collections()
print("Collections in ChromaDB:", collections)


collection = chroma_client.get_collection(name="religious_collection")

query  = "what is panchikarana?"
# 4. Perform a similarity search
results = collection.query(
    query_texts=[query], # Your search query
    n_results=10                           # Number of similar results to return
)

print(results["ids"])  # Print the IDs of the similar documents



Collections in ChromaDB: [Collection(name=religious_collection)]
[['documents\\tattva_bodha.pdf_65', 'documents\\bgita_1.pdf_871', 'documents\\bgita_1.pdf_632', 'documents\\bgita_1.pdf_539', 'documents\\bgita_1.pdf_400', 'documents\\bgita_1.pdf_906', 'documents\\atma_bodha.pdf_12', 'documents\\bgita_1.pdf_541', 'documents\\bgita_1.pdf_815', 'documents\\bgita_1.pdf_505']]


In [3]:
from langchain_ollama import ChatOllama
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_chroma import Chroma
vector_store = Chroma(collection_name="religious_collection", client=chroma_client)

def query_after_getting_matched_documents(user_query, ollama_model_name="granite4.1:3b"):
    # Create a retriever from the vector store getting top 10 similar documents
    retriever = vector_store.as_retriever(collection_name="religious_collection", search_type="similarity", search_kwargs={"k": 10})

    llm = ChatOllama(model=ollama_model_name, base_url=None)
    # ConversationalRetrievalChain wraps the LLM + retriever
    chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, return_source_documents=True)

    result = chain.invoke({"question": user_query, "chat_history":[]})
    print(result["answer"])
    matching_docs = result["source_documents"]
    print("Matching document IDs:")
    for doc in matching_docs:
        print(doc.id)

In [4]:
query = "What is panchikarana?" 
query_after_getting_matched_documents(query)

Panchikarana refers to the five stages or steps in Bhakti Yoga (devotional service) as described by Sri Chaitanya Mahaprabhu in his works, particularly in the Caitanya Caritamrta. These five stages are:

1. **Kirtan (Singing of Holy Names)**: This is the initial stage where devotees start to engage with Krishna through singing His holy names and songs. It involves chanting the names of Lord Krishna using various musical instruments and rhythmic compositions.

2. **Nadabrahma (Listening to Divine Discourses)**: In this stage, devotees begin to immerse themselves in the study of scriptures related to Lord Krishna and engage in listening to discourses that explain the nature and glories of His pastimes. This helps deepen their understanding and love for Him.

3. **Sadhana (Initiation into Devotional Practices)**: Here, devotees start performing regular devotional activities such as offering prayers, performing pujas (worship), chanting Vedic mantras, and engaging in other rituals prescrib